## Data acquisition and preprocessing (SDSS stellar sample)

In this section, we construct a clean stellar dataset from the Sloan Digital Sky Survey (SDSS) for the purpose of learning the mapping between broadband photometry and stellar physical parameters.

The workflow consists of three main steps:

1. **Querying SDSS** to retrieve photometric measurements, associated uncertainties, and spectroscopic stellar parameters (effective temperature, metallicity, surface gravity).
2. **Cleaning and filtering** the data to ensure physical consistency, remove unreliable measurements, and restrict the sample to well-behaved stellar populations.
3. **Feature engineering**, where raw magnitudes are transformed into physically meaningful colour indices, and a global photometric noise proxy is constructed.

---

### Scientific motivation

Stellar parameter inference from surveys like SDSS is fundamentally a problem of extracting physical information from noisy, indirect observables. Broadband magnitudes alone are not physically interpretable; instead, **colour indices** provide a more stable representation of stellar properties.

However, the mapping between colours and stellar parameters is:

- nonlinear (due to stellar atmosphere physics and filter responses)  
- degenerate (multiple physical parameters can produce similar colours)  
- noise-dependent (measurement uncertainties vary across observations)  

This makes careful preprocessing essential before applying any machine learning model.

---

### Goal of this pipeline

The goal of this stage is to produce a dataset that:

- retains the essential physical information (stellar parameters + colours)  
- removes unphysical or low-quality measurements  
- encodes observational uncertainty in a compact form  
- is suitable for both deterministic and probabilistic modelling  

The resulting dataset forms the basis for all subsequent regression and uncertainty quantification experiments.

## Imports

We import the core libraries needed for querying, handling, and visualising astrophysical data.

- `pandas`: tabular data manipulation  
- `numpy`: numerical computation  
- `matplotlib`: plotting  
- `astroquery.sdss`: interface to SDSS database  

These form the basic pipeline for retrieving and analysing stellar survey data.

In [ ]:
import pandas as pd
from astroquery.sdss import SDSS
import matplotlib.pyplot as plt
import numpy as np


## SDSS query: selecting a clean stellar sample

We construct an SQL query to retrieve a physically meaningful subset of stellar objects from SDSS.

### What is being selected

- **Photometry:** ugriz magnitudes  
- **Photometric errors:** measurement uncertainties in each band  
- **Stellar parameters:** effective temperature, metallicity, surface gravity  

### Selection strategy

We apply constraints to ensure data quality and physical plausibility:

- **Stellar objects only:** `p.type = 6`  
- **Valid stellar parameter ranges:** filters out unphysical or poorly constrained fits  
- **Clean photometry:** removes contaminated or unreliable sources  
- **Finite values only:** ensures no missing astrophysical parameters  
- **Uncertainty bounds:** removes extremely noisy or saturated measurements  
- **Colour constraints:** restricts to the expected stellar locus region  

### Purpose

The goal is not to maximise dataset size, but to ensure a **physically consistent and high-quality training sample** for subsequent machine learning models.

In [ ]:
query="""
SELECT DISTINCT TOP 9000 
    p.objid,
    -- Photometry
    p.u, p.g, p.r, p.i, p.z,
    -- Dereddened Photometry (Physical)
    p.dered_u, p.dered_g, p.dered_r, p.dered_i, p.dered_z,
    -- Extinction values
    p.extinction_u, p.extinction_g, p.extinction_r, p.extinction_i, p.extinction_z,
    -- Photometric uncertainties
    p.err_u, p.err_g, p.err_r, p.err_i, p.err_z,
    -- Stellar parameters
    s.teffadop AS Teff,
    s.fehadop AS feh,
    s.loggadop AS logg
FROM PhotoObj AS p
INNER JOIN sppParams AS s 
    ON p.objid = s.bestobjid
WHERE (p.objid % 100) = 42
    AND p.type = 6  -- Added the missing 'AND' here
    AND s.teffadop BETWEEN 3200 AND 9500 
    AND s.fehadop BETWEEN -3.5 AND 0.5
    AND s.loggadop BETWEEN 0.5 AND 5.0
    AND p.clean = 1
    AND p.err_u BETWEEN 0.001 AND 0.5
    AND p.err_g BETWEEN 0.001 AND 0.5
    AND p.err_r BETWEEN 0.001 AND 0.5
    AND p.err_i BETWEEN 0.001 AND 0.5
    AND p.err_z BETWEEN 0.001 AND 0.5
    AND (p.dered_u - p.dered_g) BETWEEN -0.5 AND 5.0
    AND (p.dered_g - p.dered_r) BETWEEN -0.2 AND 2.5
ORDER BY p.objid
"""
   

## Query execution and data loading

We execute the SDSS query and convert the result into a Pandas DataFrame for analysis.

At this stage, each row corresponds to a single star with:

- photometric measurements (u, g, r, i, z)  
- uncertainties in each band  
- spectroscopic stellar parameters (Teff, [Fe/H], log g)  

We inspect the first rows to verify that the data structure is correct and complete.

In [ ]:
result_table = SDSS.query_sql(query, data_release=17)

if result_table is None:
    raise RuntimeError(
        "SDSS query returned no data. "
        "Check internet connection or SDSS server status."
    )

df = result_table.to_pandas()

print(f"Retrieved {len(df)} stars")
df.head()


## Basic dataset statistics

We compute summary statistics of the dataset to understand:

- ranges of photometric magnitudes  
- distribution of stellar parameters  
- presence of outliers or extreme values  

This step provides a quick sanity check before feature engineering and modelling.

In [ ]:
df.describe()

## Feature engineering: colour indices and noise propagation

We transform raw magnitudes into physically meaningful **colour indices**:

- $u-g$, $g-r$, $r-i$, $i-z$

These encode differences in flux between bands and are more directly related to stellar properties than raw magnitudes.

---

### Uncertainty propagation

We additionally construct a scalar photometric quality metric, `sigma_phot`, defined as the root-mean-square of the reported SDSS magnitude uncertainties across the five photometric bands:

$
\sigma_{\mathrm{phot}}
=
\sqrt{
\frac{
\sigma_u^2+\sigma_g^2+\sigma_r^2+\sigma_i^2+\sigma_z^2
}{5}
}
$

This quantity provides a compact estimate of the overall observational noise level associated with each source.
Unlike propagated colour uncertainties, this metric avoids double-counting shared band errors arising from correlated colour indices.

In [ ]:
df_model = df.copy()

# ---------------------------------
# Colour features
# ---------------------------------
df_model["u_g"] = df_model["dered_u"] - df_model["dered_g"]
df_model["g_r"] = df_model["dered_g"] - df_model["dered_r"]
df_model["r_i"] = df_model["dered_r"] - df_model["dered_i"]
df_model["i_z"] = df_model["dered_i"] - df_model["dered_z"]

# ---------------------------------
# Individual colour uncertainties
# ---------------------------------
df_model["sigma_u_g"] = np.sqrt(
    df_model["err_u"]**2 +
    df_model["err_g"]**2
)

df_model["sigma_g_r"] = np.sqrt(
    df_model["err_g"]**2 +
    df_model["err_r"]**2
)

df_model["sigma_r_i"] = np.sqrt(
    df_model["err_r"]**2 +
    df_model["err_i"]**2
)

df_model["sigma_i_z"] = np.sqrt(
    df_model["err_i"]**2 +
    df_model["err_z"]**2
)

# ---------------------------------
# Scalar photometric quality metric
# ---------------------------------
df_model["sigma_phot"] = np.sqrt(
    (
        df_model["err_u"]**2 +
        df_model["err_g"]**2 +
        df_model["err_r"]**2 +
        df_model["err_i"]**2 +
        df_model["err_z"]**2
    ) / 5
)

## Colour–colour structure of stellar populations

This scatter plot visualises the stellar locus in colour–colour space.

- X-axis: $g - r$ (temperature-sensitive colour index)  
- Y-axis: $u - g$ (sensitive to metallicity and line blanketing)  
- Colour: effective temperature ($T_{\rm eff}$)  

### Interpretation

- Hot stars appear in the blue/low-colour region  
- Cool stars appear in the red/high-colour region  
- The curved structure reflects the **nonlinear stellar locus**

### Physical insight

The observed curvature is not noise; it arises from:

- stellar atmosphere physics  
- filter transmission effects  
- metallicity-dependent absorption features  

This already suggests that purely linear models may struggle to capture the full structure of the stellar locus across broad parameter ranges.

In [ ]:
plt.figure(figsize=(7, 6))

sc = plt.scatter(
    df_model["g_r"],
    df_model["u_g"],
    c=df_model["Teff"],
    cmap="RdYlBu_r",
    s=2,
    alpha=0.7
)

plt.colorbar(sc, label=r"$T_{\rm eff}$ [K]")

plt.xlabel(r"$g-r$")
plt.ylabel(r"$u-g$")

plt.title("SDSS Stellar Locus")

plt.tight_layout()
plt.show()

### Final dataset preparation

We remove:

- identifiers (`objid`)  
- raw magnitudes (replaced by colours)  
- individual band uncertainties (replaced by propagated noise metrics)  

This results in a clean modelling dataset containing:

- colour features  
- stellar parameters  
- global noise estimate  

The goal is to provide a compact representation suitable for statistical learning while preserving physical information.

In [ ]:
cleandf = df_model.drop(columns=[
    "objid",
    "u", "g", "r", "i", "z",
    "dered_u", "dered_g", "dered_r", "dered_i", "dered_z",
    "extinction_u", "extinction_g", "extinction_r",
    "extinction_i", "extinction_z",
    "err_u", "err_g", "err_r", "err_i", "err_z",
    "sigma_u_g", "sigma_g_r", "sigma_r_i", "sigma_i_z"
])

cleandf.head()


## Saving the processed dataset

We export the cleaned dataset to a CSV file for reuse in downstream notebooks.

This ensures:

- reproducibility of experiments  
- separation between data acquisition and modelling  
- efficient reuse without re-querying SDSS  

The resulting dataset is now ready for machine learning experiments on stellar parameter inference.

In [ ]:
cleandf.to_csv('./data/sdss_processed_colors_v1.csv', index=False)